In [ ]:
from network_expansion import *
from externalize_results import *
import networkx as nx
import matplotlib.pyplot as plt
from networkx.algorithms import bipartite
import numpy as np
from matplotlib.colors import Normalize, to_hex, BoundaryNorm
import matplotlib
import netwulf as nw
from netgraph import Graph, get_multipartite_layout

# Expansions from primordial seed sets
write_outputs = True
max_generation = 1000
name = 'primordial'

# define chemical scope
reaction_scope = get_sides(import_reaction_file())
compound_scope = set.union(*reaction_scope.stack())
seed_scope = seed_candidates.loc[seed_candidates.index.isin(compound_scope)]
seeds = list(seed_scope.index)

# run network expansion
result = expand_from_seed_set(seeds=seeds, reactions=reaction_scope, max_gen=max_generation)
write_expansion_csv(result, filename=name, destination_dir='Output/'+name+'/')
result = generate_annotated_bipartite_edge_list(result)
# summary = result
transition_gen = result['chiral_fraction'].where(lambda x: x>0.5).idxmin()
print(f"Transition Step: {transition_gen}")
#transition_gen = 10
transition_edge_df = result['bipartite_edgelist'].query('step <= @transition_gen').copy(deep=True)
print(transition_edge_df)
networks = {i: nx.from_pandas_edgelist(transition_edge_df[transition_edge_df[i]], 
        source='source', target='target', edge_attr=True, create_using=nx.DiGraph())
        for i in ['impactful_precursor','directed']} #immediate precursor
compound_nodes = {i: [l for l in bipartite.sets(networks[i]) if list(l)[0].startswith('C')][0] for i in networks.keys()}
networks_uni = {i: bipartite.projected_graph(networks[i], nodes=compound_nodes[i]) for i in networks.keys()}


In [ ]:
def gna(G, attr): return nx.get_node_attributes(G, attr)

def colorize_Series(S, cmap='cet_CET_C6s'):
    # S should be list-like, or a dict, or a series.... of numbers.
    color_map = matplotlib.colormaps[cmap]
    def float_norm(x): return(Normalize(vmin=x.min(), vmax=x.max()))
    def boundary_norm(x): return(BoundaryNorm(boundaries=list(range(x.min(), x.max()+1, 1)), ncolors=color_map.N))
    if type(S)!=pd.Series:
        S = pd.Series(S)
    if S.dtype==pd.Float64Dtype():
        c_ser = pd.Series(map(tuple, color_map(float_norm(S)(list(S.values)))), index=S.index)
        print("FLOAT")
    elif S.dtype==pd.Int64Dtype():
        c_ser = pd.Series(map(tuple, color_map(boundary_norm(S)(list(S.values)))), index=S.index)
        print("INT")
    else:
        S = pd.Series(S.factorize()[0])/(pd.Series(S.factorize()[0]).max())
        c_ser = pd.Series(map(tuple, color_map(float_norm(S)(list(S.values)))), index=S.index)
        c_ser = c_ser.apply(to_hex)
        print(c_ser)
    return(c_ser)

In [ ]:
gen_cmap = 'Greys'

attributes = result['arrival_step'].filter(like='C').to_frame(name='gen').query('gen.lt(@transition_gen+1)').rename_axis(None)
attributes = attributes.join(chiral_labels['is_chiral'], how='outer')
attributes['is_chiral_color'] = attributes['is_chiral'].map({True: '#34CCFF', False: '#FFC012'}).fillna("#FFFFFF")
attributes['gen_color'] = colorize_Series(attributes['gen'], cmap=gen_cmap).tolist()


In [ ]:
size_nodes_by = 'degree'
max_node_size = 25

G = networks_uni['directed'].copy()
node_sizes = pd.Series({i:(np.log2(j+2))**1.5 for i,j in G.degree()})
nx.set_node_attributes(G, node_sizes, 'size')
nx.set_node_attributes(G, attributes['is_chiral_color'], 'group')

print(G)
fig = plt.figure(figsize=(12,12), layout='constrained')
stylized_network, config = nw.visualize(G, config={'node_label_color': '#000000', 'node_gravity': 0.0, 
                                'link_alpha': 0.005, 'link_color': '#000000', 'zoom': 0.63, 'node_charge': -24,
                                'scale_node_size_by_strength': False, 'node_size_variation': 1,
                                'node_size': max_node_size, 'link_distance':0, 'node_stroke_width': 2})
fig, ax = nw.draw_netwulf(stylized_network)

a = stylized_network.copy()
a['nodeStrokeWidth'] = 0.0
for i in list(range(len(a['nodes']))):
    a['nodes'][i]['radius'] = a['nodes'][i]['radius'] * 0.4
    a['nodes'][i]['color'] = attributes['gen_color'].loc[a['nodes'][i]['id']]

nw.draw_netwulf(a, fig=fig, ax=ax, draw_links=False)

plt.show()

In [ ]:
#attributes['test'] = (attributes['precursor_out_degree_chiral']/(attributes['is_chiral']-0.5)).filter(like='C').sort_values()
#sortby = ['is_chiral', 'test']
show = 'impactful_precursor'
rotate = False

G1 = networks_uni[show] #using impactful_precursor from network generation

#Remove G nodes that are not in attributes df
nodes_to_keep = attributes.index.to_list()
nodes_to_remove = [n for n in nodes_to_keep if n not in G1.nodes()]
print(nodes_to_remove)
G1.remove_nodes_from(nodes_to_remove)

nx.set_node_attributes(G1, attributes['is_chiral_color'], 'is_chiral_color')

if "C21389" in G1.nodes():
    print("Found C21389")
print(G1)

layers = [list(i) for i in attributes.loc[list(G1.nodes())].groupby(by='gen').groups.values()][:-1]
for layer in layers:
    if "C21389" in layer:
        print(f"Found C21389")
    print(layer)
    print(len(layer))


pos = get_multipartite_layout(edges=list(G1.edges), layers=layers, reduce_edge_crossings=True)
#for i in layers[-1]: del pos[i] # remove last layer
for n in pos.keys(): pos[n][0] = (pos[n][0]+0.2)**2 # spread out later gens to reduce extreme angles 
if rotate: pos = {i:j[::-1] for i,j in pos.items()}

print(pos)
#print(pos["C21389"])

fig, ax = plt.subplots(figsize=(15,15), layout='constrained')
nx.draw_networkx(G1.subgraph(pos.keys()), pos=pos, ax=ax, edge_color=[0.5,0.5,0.5,0.5],
                  node_color=list(gna(G1.subgraph(pos.keys()), 'is_chiral_color').values()),  
                  edgecolors='k', width=0.01, arrows=False,
                  linewidths=0.2, node_size=100, node_shape='s', alpha=0.99,
                  with_labels=False)

# nx.draw_networkx_nodes(G1, pos=pos, ax=ax, node_color=list(gna(G1.subgraph(pos.keys()), 'is_chiral_color').values()), 
#                        node_size=100, node_shape='s', alpha=1.00) #draw nodes
# [nx.draw_networkx_edges(G1, pos=pos, arrows=False, width=0.05,
#                         alpha=0.05)] #loop through edges and draw them


plt.legend(loc='lower left', borderpad=8, frameon=False, handles=[plt.Line2D([0], [0], marker='s', color='w', markeredgecolor='k', markeredgewidth=0.2, markerfacecolor='#34CCFF', markersize=20, label='Chiral'),
                    plt.Line2D([0], [0], marker='s', color='w', markerfacecolor='#FFC012', markeredgecolor='k', markeredgewidth=0.2, markersize=20, label='Achiral')], prop={'size': 20})
#plt.savefig('fig3_expansion_'+show+'_edges_minimal_crossings.svg', format='svg', facecolor='w', edgecolor='w')
plt.show()